# 2. Train a masked protein language model
- **Input:** Natural UniRef50 representative sequences, without functional labels.
- **Model:** Amino acid and position embeddings, a bidirectional transformer encoder, and an amino acid prediction head.
- **Training objective:** Predict original residues at selected positions from a corrupted sequence.

Train a small BERT-style encoder from scratch, compare it with an amino acid frequency baseline, and inspect masked-residue predictions.

## Setup
Select **Runtime → Change runtime type → T4 GPU**, then run the cells below.

In [ ]:
%pip -q install lightning==2.5.5
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
import lightning.pytorch as pl
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.callbacks import ModelCheckpoint
import pandas as pd
import matplotlib.pyplot as plt

pl.seed_everything(7, workers=True)
torch.set_num_threads(4)

## 1. Prepare natural protein sequences
Load 4,000 natural UniRef50 representative sequences, 40–120 residues long. Shuffle the sequences and reserve separate validation and test sets. Amino acids become integer IDs; shorter sequences are padded to the same length.

In [ ]:
AA = 'ACDEFGHIKLMNPQRSTVWY'
aa_to_id = {aa: i for i, aa in enumerate(AA)}
MASK, PAD, IGNORE = 20, 21, -100

In [ ]:
sequence_source = 'https://raw.githubusercontent.com/RomeroLab/BME305-protein-network-training/main/data/uniref50_small.csv'
df = pd.read_csv(sequence_source).sample(frac=1, random_state=7).reset_index(drop=True)
encoded = [torch.tensor([aa_to_id[aa] for aa in sequence]) for sequence in df.sequence]
x = nn.utils.rnn.pad_sequence(encoded, batch_first=True, padding_value=PAD)
print(f'{len(df):,} natural sequences; padded token tensor: {tuple(x.shape)}')
display(df[['cluster_id', 'length', 'sequence']].head())

In [ ]:
batch_size = 128
n_train, n_val = int(0.8 * len(x)), int(0.1 * len(x))
x_train = x[:n_train]
x_val = x[n_train:n_train+n_val]
x_test = x[n_train+n_val:]

## 2. Create the masked prediction task
Select 15% of real residues, with at least one selected residue per sequence. Of these, replace 80% with `[MASK]`, 10% with a random amino acid, and leave 10% unchanged. Labels retain the original residues; `-100` excludes unselected positions and padding from the loss.

Training uses new corruptions each batch. Validation and test corruptions stay fixed so evaluation is comparable across epochs.

In [ ]:
mask_probability = 0.15

def mask_tokens(tokens, probability=0.15, generator=None):
    selected = (torch.rand(tokens.shape, generator=generator) < probability) & (tokens != PAD)
    selected[~selected.any(dim=1), 0] = True
    labels = tokens.clone()
    labels[~selected] = IGNORE
    corrupted = tokens.clone()
    choices = torch.rand(tokens.shape, generator=generator)
    corrupted[selected & (choices < 0.8)] = MASK
    random_ids = torch.randint(20, tokens.shape, generator=generator)
    random_positions = selected & (choices >= 0.8) & (choices < 0.9)
    corrupted[random_positions] = random_ids[random_positions]
    return corrupted, labels

def training_batch(sequences):
    return mask_tokens(torch.stack(sequences), probability=mask_probability)

val_tokens, val_labels = mask_tokens(x_val, mask_probability, torch.Generator().manual_seed(17))
test_tokens, test_labels = mask_tokens(x_test, mask_probability, torch.Generator().manual_seed(27))
train_loader = DataLoader(x_train, batch_size=batch_size, shuffle=True, collate_fn=training_batch)
val_loader = DataLoader(TensorDataset(val_tokens, val_labels), batch_size=batch_size)
test_loader = DataLoader(TensorDataset(test_tokens, test_labels), batch_size=batch_size)
corrupted, labels = mask_tokens(x_train[:1], mask_probability, torch.Generator().manual_seed(37))
print('Original: ', df.sequence.iloc[0])
print('Corrupted:', ' '.join('[MASK]' if i == MASK else AA[i] for i in corrupted[0] if i != PAD))
print('Labels:   ', ' '.join('.' if i == IGNORE else AA[i] for i in labels[0, :len(df.sequence.iloc[0])]))

## 3. Define the transformer and training steps
Add amino acid and learned position embeddings, then apply self-attention. Each real position can attend to both sides of the sequence. The padding mask excludes padded positions from attention. The prediction head outputs 20 amino acid logits per position.

Cross-entropy compares the predicted amino acid distribution with the original residue at each selected position. Unselected residues and padding are excluded from the loss.

In [ ]:
class MaskedProteinModel(pl.LightningModule):
    def __init__(self, length, d_model=48, n_heads=4, n_layers=2, learning_rate=0.003):
        super().__init__()
        self.save_hyperparameters()
        self.amino_acids = nn.Embedding(22, d_model, padding_idx=PAD)
        self.positions = nn.Embedding(length, d_model)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=2*d_model, dropout=0.1, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        # Encoder layers are cloned; initialize their matrix weights independently.
        for encoder_layer in self.encoder.layers:
            for parameter in encoder_layer.parameters():
                if parameter.ndim > 1:
                    nn.init.xavier_uniform_(parameter)
        self.head = nn.Linear(d_model, 20)

    def forward(self, tokens):
        positions = torch.arange(tokens.shape[1], device=tokens.device)
        embedded = self.amino_acids(tokens) + self.positions(positions)
        return self.head(self.encoder(embedded, src_key_padding_mask=tokens.eq(PAD)))

    def shared_step(self, batch, stage):
        tokens, labels = batch
        logits = self(tokens)
        loss = F.cross_entropy(logits.reshape(-1, 20), labels.reshape(-1), ignore_index=IGNORE)
        self.log(stage + '_loss', loss, on_step=False, on_epoch=True, prog_bar=True, batch_size=(labels != IGNORE).sum().item())
        return loss

    def training_step(self, batch, batch_idx):
        return self.shared_step(batch, 'train')

    def validation_step(self, batch, batch_idx):
        return self.shared_step(batch, 'val')

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)

In [ ]:
epochs = 5
model = MaskedProteinModel(length=x.shape[1], d_model=48, n_heads=4, n_layers=2, learning_rate=0.003)
print(f'{sum(p.numel() for p in model.parameters()):,} trainable parameters')
print('Input shape:', tuple(corrupted.shape), '→ output shape:', tuple(model(corrupted).shape))

## 4. Train and inspect learning curves
Lightning optimizes the weights using masked-residue loss. Validation uses the same corrupted sequences each epoch, and the lowest validation loss selects the model.

In [ ]:
logger = CSVLogger('training_logs', name='protein_model')
checkpoint = ModelCheckpoint(monitor='val_loss', mode='min', save_top_k=1)
trainer = pl.Trainer(max_epochs=epochs, accelerator='auto', devices=1, logger=logger, callbacks=[checkpoint], enable_model_summary=False, num_sanity_val_steps=0)
trainer.fit(model, train_loader, val_loader)
model = type(model).load_from_checkpoint(checkpoint.best_model_path).cpu().eval()

In [ ]:
metrics = pd.read_csv(logger.log_dir + '/metrics.csv')
for column in ['train_loss', 'val_loss']:
    values = metrics.dropna(subset=[column])
    plt.plot(values.epoch + 1, values[column], marker='o', label=column)
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.show()

## 5. Compare with an amino acid frequency baseline
Estimate amino acid frequencies from training residues, excluding padding. This baseline ignores sequence context and position.

Compare cross-entropy and accuracy on the same selected test residues. Compare overall selected-residue accuracy with accuracy at `[MASK]` positions, where the original residue is hidden.

In [ ]:
counts = torch.bincount(x_train[x_train != PAD], minlength=20).float() + 1
amino_acid_probabilities = counts / counts.sum()
with torch.inference_mode():
    logits = torch.cat([model(tokens) for tokens, labels in test_loader])
selected = test_labels != IGNORE
hidden = selected & (test_tokens == MASK)
predicted_ids = logits.argmax(dim=-1)
baseline_ids = amino_acid_probabilities.argmax().expand_as(x_test)
model_ce = F.cross_entropy(logits[selected], test_labels[selected]).item()
baseline_ce = -amino_acid_probabilities[test_labels[selected]].log().mean().item()
results = pd.DataFrame({'Model': ['Transformer', 'Amino acid frequencies'], 'Cross-entropy': [model_ce, baseline_ce], 'Selected-residue accuracy': [(predicted_ids[selected] == test_labels[selected]).float().mean().item(), (baseline_ids[selected] == test_labels[selected]).float().mean().item()], 'MASK-only accuracy': [(predicted_ids[hidden] == test_labels[hidden]).float().mean().item(), (baseline_ids[hidden] == test_labels[hidden]).float().mean().item()]})
print(f'{selected.sum().item():,} selected test residues; {hidden.sum().item():,} replaced with [MASK]')
display(results.round(3))

## 6. Inspect one masked residue
Choose a held-out sequence and a position (numbered from 1, within that sequence). Replace the residue with `[MASK]` and compare the model's distribution with the amino acid frequency baseline. Inspect the probability assigned to the original residue and the alternatives favored by the model.

In [ ]:
sequence_index = 0
position = 10
query = x_test[sequence_index:sequence_index+1].clone()
query[0, position-1] = MASK
with torch.inference_mode():
    probabilities = model(query)[0, position-1].softmax(dim=-1)
comparison = pd.DataFrame({'Amino acid': list(AA), 'Transformer': probabilities.numpy(), 'Amino acid frequencies': amino_acid_probabilities.numpy()})
print(f'Hidden residue at position {position}: {AA[x_test[sequence_index, position-1]]}')
display(comparison.sort_values('Transformer', ascending=False).head(5).round(3))
comparison.set_index('Amino acid').plot.bar(figsize=(9, 3))
plt.ylabel('Probability'); plt.tight_layout(); plt.show()

## Explore
Change one setting—mask probability, embedding width, encoder layers, or learning rate—and retrain with the same sequence split. Embedding width must be divisible by the number of attention heads.

- Does the transformer improve on amino acid frequencies? What can neighboring residues contribute?
- How do mask probability and model size affect validation loss?
- Why would broader sequence coverage and homology-aware test splits matter for a larger protein language model?

## Save results
Save the model and token definitions for later sequence experiments.

In [ ]:
torch.save({'model_state': model.state_dict(), 'hyperparameters': dict(model.hparams), 'amino_acids': AA, 'mask_token': MASK, 'padding_token': PAD}, 'uniref50_masked_model.pt')
results.to_csv('masked_model_test_metrics.csv', index=False)

## References
[UniRef](https://www.uniprot.org/help/uniref) · [Sequence dataset details](https://github.com/RomeroLab/BME305-protein-network-training/blob/main/data/uniref50_manifest.json) · [LightningModule](https://lightning.ai/docs/pytorch/stable/common/lightning_module.html) · [TransformerEncoder](https://docs.pytorch.org/docs/stable/generated/torch.nn.TransformerEncoder.html) · [BERT](https://arxiv.org/abs/1810.04805)